# Advanced Certification Programme in Agentic and Generative AI
## A Programme by IISc and TalentSprint
## Additional Notebook: **RAG - Retrieval Augmented Generation**

(Without OpenAI API)

## Learning Objectives

At the end of the experiment, you will be able to:

* Understand the concept of RAG and its advantages over standalone LLMs
* Build a RAG pipeline using LangChain
* Load, preprocess, and chunk documents for efficient retrieval
* Generate embeddings and store them in a vector database (Chroma)
* Perform semantic search to retrieve relevant information based on user queries
* Addressing Diversity and Specificity during retrieval
* Generate accurate, context-aware responses using retrieved data
* Design and test real-world use cases like HR Policy Assistant and Product Support Chatbot


## Introduction

#### **What is RAG (Retrieval-Augmented Generation)?**

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a Large Language Model (LLM) by allowing it to **retrieve relevant information from external documents** before generating a response.

In simple terms:

> **RAG = Search (retrieve) + Generate (LLM answer)**

---

#### **Why do we need RAG?**

LLMs by themselves:

* Don’t know your **company data**
* Can give **generic or incorrect answers (hallucinations)**
* Are limited to their **training knowledge**

RAG solves this by:

* Connecting LLMs to **your documents**
* Producing **accurate, grounded responses**
* Reducing hallucinations

---

#### **How RAG Works (Simple Flow)**

1. **User asks a question**
2. System **searches relevant chunks** from documents
3. Retrieved content is **sent to the LLM as context**
4. LLM generates a **final, grounded answer**

---

#### **Key Components**

* **Document Loader** → Load PDFs, docs, CSVs
* **Chunking** → Break text into smaller pieces
* **Embeddings** → Convert text into vectors
* **Vector Database** (e.g., FAISS, Chroma) → Store & search
* **Retriever** → Fetch relevant chunks
* **LLM** → Generate final answer

---

#### **Example**

Without RAG:

> “What is my company’s leave policy?”
>
> → ❌ Generic answer

With RAG:

> Same question
>
> → ✅ Answer pulled from **your HR policy document**

---

#### **When to Use RAG**

* Internal knowledge assistants
* Customer support bots
* Document Q&A systems
* Enterprise search

---


> **RAG diagram:**
>
> <img src='https://drive.google.com/uc?id=1sCVvpsmtZEU1WSK1FFGMGHbEjrgtCNLi'>


The above diagram illustrates the difference between a standard LLM workflow and a RAG pipeline:

* **without RAG**, the user’s question is sent directly to the LLM, which generates an answer based only on its pre-trained knowledge, often leading to generic or incorrect responses;

* **with RAG**, the system

    * first retrieves relevant information from a knowledge base (**retrieval**),
    * adds this context to the original question (**augmentation**), and
    * then passes it to the LLM to generate a more accurate, context-aware answer (**generation**), thereby reducing hallucinations and improving reliability.



---
---

> **Vector Store and Retrieval:**
>
> <img src='https://drive.google.com/uc?id=1_zX5gtSNrV8Qdx7Nz4_gMR8dCwvxCDS7' width=750px>


The above diagram illustrates the end-to-end workflow of a vector store–based retrieval system in a RAG pipeline:

* first, data from sources like PDFs, URLs, or databases is loaded and converted into documents,
* which are then split into smaller chunks for better processing;
* these chunks are transformed into embeddings and stored in a vector database (vector store).

During retrieval, when a user submits a query, it is also converted into an embedding and compared against the stored vectors to find the most relevant chunks; these retrieved pieces of information are then added to the prompt and passed to the LLM, which uses this context to generate a more accurate and context-aware answer.


---
---

> **Embedding Model:**
>
> <img src='https://drive.google.com/uc?id=1HnvjGJ4HmpS-0wndpH-Q8cKMwIwWkTUe'>


The above diagram shows how an **embedding model converts text into a numerical vector representation**: input text is processed by the embedding model, which transforms it into a list of numbers (vector) that captures its semantic meaning; these vectors enable similarity comparison, allowing systems like RAG to find and retrieve contextually relevant information efficiently.


---
---

> **Retrieval in Action:**
>
> <img src='https://drive.google.com/uc?id=1ry2TWFsewwqYP3Lw9muuPmbyuQqXwnYV' width=800px>


The above diagram shows how retrieval works in practice:

* during the **create phase**, documents are split into smaller chunks, converted into embeddings, and stored in a vector store along with their original text;

* during the **index/query phase**, a user query is also embedded and compared against all stored vectors to find the most similar chunks, and the top relevant results are selected to be used as context for generating the final answer.


---
---

> **Example workflow with embedding model:**
>
><br>
>
> <img src='https://drive.google.com/uc?id=1zTuMMX54L2HrnmCYktTxVfMVrkIz8w15' width=600px>


The above diagram shows a complete RAG workflow:

* proprietary data is first converted into embeddings and stored in a vector database,
* while a user query is also embedded and used to search for the most relevant documents;
* the top matching results are combined with the original question to form an enriched prompt,
* which is then passed to the LLM to generate a context-aware and accurate answer.

---
---

### Install Dependencies

In [ ]:
%%capture
!pip -q install langchain==0.3.27
!pip -q install openai==2.3.0
!pip -q install langchain-core==0.3.79
!pip -q install langchain-community==0.3.31
!pip -q install sentence-transformers==5.1.1
!pip -q install langchain-huggingface==0.3.1
!pip -q install langchain-experimental==0.3.4
!pip -q install langchainhub==0.1.21
!pip -q install langchain-openai==0.3.35
!pip -q install langchain-chroma==0.2.6
!pip -q install chromadb==1.1.1
!pip -q install pymupdf==1.27.2.3

### Import Required Packages

In [ ]:
# Import built-in libraries
import os                  # Used for environment variables (e.g., API keys)
import numpy as np         # Used for numerical operations (often useful in embeddings)

# Document loader to read PDF files and convert them into LangChain document format
from langchain_community.document_loaders import PyMuPDFLoader

# LLM interface to interact with OpenAI chat models (used for response generation)
from langchain_openai import ChatOpenAI

# Vector store (database) to store embeddings and perform similarity search
from langchain_chroma import Chroma

# Used to define structured prompts (input template for LLM)
from langchain_core.prompts import PromptTemplate

# Parses LLM output into a simple string format
from langchain_core.output_parsers import StrOutputParser

# Enables chaining of components (like passing input through multiple steps in pipeline)
from langchain.schema.runnable import RunnablePassthrough

### **Provide your OpenAI API key**

In [ ]:
# Read OpenAI key from Colab Secrets

import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

### Initialize LLM

In [ ]:
# Initialize LLM

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",   # Groq-supported model
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
# Test the LLM
response = llm.invoke("What is 2 plus 3?")
print(response.content)

### **Loading the documents**

In this notebook, we will be using the below documents:

1. **HR Policy – Employee Handbook.pdf**

    This document contains the organization’s internal policies and guidelines related to employees, including topics such as leave policies, working hours, code of conduct, reimbursement rules, and benefits. It serves as a centralized reference for employees to understand company procedures and compliance requirements.

---

2. **Product XSound Pro Headphones.pdf**

    This document is a product manual and support guide for *XSound Pro Headphones*, covering features, specifications, setup instructions, troubleshooting steps, warranty details, and usage guidelines. It helps users understand the product and resolve common issues effectively.

---

The documents will be loaded using LangChain's [PDF Loader](https://docs.langchain.com/oss/javascript/integrations/document_loaders/file_loaders/pdf).

The documents are loaded as `Document` objects which comprises of `page_content` and `metadata`.

* **page_content:** The actual text content extracted from each page of the document.
* **metadata:** Additional information about the document/page (e.g., source file, page number, etc.).


In [ ]:
#@title Run this cell to Download the Document PDF files

from IPython.display import clear_output

!wget https://raw.githubusercontent.com/MLOPS-test/Artifacts/main/datasets/HR%20Policy%20–%20Employee%20Handbook.pdf
!wget https://raw.githubusercontent.com/MLOPS-test/Artifacts/main/datasets/Product%20XSound%20Pro%20Headphones.pdf

clear_output()

!ls | grep '.pdf'

In [ ]:
# UPLOAD the Docs first to this notebook, then run this cell

from langchain_community.document_loaders import PyMuPDFLoader

# Load PDF
loaders = [
    PyMuPDFLoader("/content/HR Policy – Employee Handbook.pdf"),
    PyMuPDFLoader("/content/Product XSound Pro Headphones.pdf"),
    PyMuPDFLoader("/content/Product XSound Pro Headphones.pdf"),    # <-- Loading duplicate documents on purpose
]

docs = []
for loader in loaders:
    docs.extend(loader.load())


In [ ]:
len(docs)        # 9 pages were there in total from above documents

In [ ]:
docs[0]

In [ ]:
# Document-1, first page

print(docs[0].page_content)

In [ ]:
# Document-2, first page

print(docs[1].page_content)

### **Splitting of Document**

This refers to breaking large documents into smaller, manageable chunks using [LangChain Text Splitters](https://reference.langchain.com/python/langchain-text-splitters).

This improves retrieval accuracy in RAG systems by ensuring that only the most relevant portions of text are embedded, stored, and retrieved, rather than processing entire documents at once.

---

**RecursiveCharacterTextSplitter** is one of the LangChain text splitters that breaks documents into smaller chunks by recursively using a hierarchy of separators (such as paragraphs, sentences, and characters).

The below code initializes a **RecursiveCharacterTextSplitter** that divides documents into chunks of 300 characters with an overlap of 30 characters between consecutive chunks, helping preserve context across splits and improving retrieval accuracy in RAG pipelines.




In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 30
)

In [ ]:
# Split documents into Chunks
splits = text_splitter.split_documents(docs)

print(f"Total Chunks: {len(splits)}\n")
print(f"First Chunk length: {len(splits[0].page_content)}\n")

print("First Chunk Page content: ")
splits[0].page_content

In [ ]:
splits[0]

### **Embeddings**

Let's take our splits and embed them.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Initialize an Embedding Model

from langchain_huggingface import HuggingFaceEmbeddings

modelPath ="mixedbread-ai/mxbai-embed-large-v1"                  # Model card: https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1
                                                                 # Find other Emb. models at: https://huggingface.co/spaces/mteb/leaderboard

# Create a dictionary with model configuration options, specifying to use the CPU for computations
model_kwargs = {'device': device}      # cuda/cpu

# Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
encode_kwargs = {'normalize_embeddings': False}

embedding =  HuggingFaceEmbeddings(
    model_name=modelPath,     # Provide the pre-trained model's path
    model_kwargs=model_kwargs, # Pass the model configuration options
    encode_kwargs=encode_kwargs # Pass the encoding options
)

#### **Understanding similarity search with a toy example**

In [ ]:
# Check the similarity among these sentences

sentence1 = "i like dogs"
sentence2 = "i like cats"
sentence3 = "the weather is ugly, too hot outside"

In [ ]:
# Generate Embeddings for each sentence

embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [ ]:
# Embedding Dimension

len(embedding1), len(embedding2), len(embedding3)

In [ ]:
embedding1[:10]

In [ ]:
# Function to calculate the Cosine Similarity between vectors

import numpy as np

def cosine_similarity(vector1, vector2):
    # Ensure that the vectors are numpy arrays
    vector1 = np.array(vector1)
    vector2 = np.array(vector2)

    # Calculate the dot product of the vectors
    dot_product = np.dot(vector1, vector2)

    # Calculate the magnitude (norm) of the vectors
    norm_vector1 = np.linalg.norm(vector1)
    norm_vector2 = np.linalg.norm(vector2)

    # Compute cosine similarity
    if norm_vector1 == 0 or norm_vector2 == 0:
        return 0  # Avoid division by zero
    return dot_product / (norm_vector1 * norm_vector2)


In [ ]:
cosine_score_1_2 = cosine_similarity(embedding1, embedding2)
cosine_score_1_3 = cosine_similarity(embedding1, embedding3)
cosine_score_2_3 = cosine_similarity(embedding2, embedding3)

In [ ]:
print(f"{'Sentence 1':<15} | {'Sentence 2':<40} | {'Cosine Similarity':<10}")
print(f"{'='*90}")
print(f"{sentence1:<15} | {sentence2:<40} | {cosine_score_1_2}")
print(f"{sentence1:<15} | {sentence3:<40} | {cosine_score_1_3}")
print(f"{sentence2:<15} | {sentence3:<40} | {cosine_score_2_3}")

A **higher value of cosine similarity** indicates that two vectors (e.g., embeddings of text) are **more similar in meaning or context**.

Since cosine similarity measures the angle between vectors,
* a value closer to **1** means the vectors are pointing in the same direction (high similarity),
* around **0** means they are unrelated, and
* closer to **-1** means they are opposite in meaning.

In RAG systems, higher cosine similarity helps identify the most relevant document chunks for a given query.


### **Vectorstores**

**Vectorstores** are specialized databases used to *store embeddings* (numerical representations of text) and enable efficient similarity search.

They allow RAG systems to quickly retrieve the most relevant document chunks based on a user query, forming the foundation for accurate and context-aware responses.

---

**Chroma** is an open-source, lightweight vector database used to store and retrieve embeddings for similarity search.

The below code creates a **Chroma vector database** by storing the document splits along with their embeddings in a specified directory (`docs/chroma/`); it also ensures any previous data is cleared before creation and verifies the number of stored vectors, which should match the number of document chunks.

To know more about Chroma integration in LangChain, refer [here](https://python.langchain.com/docs/integrations/vectorstores/chroma/).



In [ ]:
from langchain_chroma import Chroma       # Light-weight vectorstore

In [ ]:
persist_directory = 'docs/chroma/'
!rm -rf ./docs/chroma  # remove old database files if any

In [ ]:
vectordb = Chroma.from_documents(
    documents=splits,                    # splits we created earlier
    embedding=embedding,
    persist_directory=persist_directory, # save the directory
)

In [ ]:
print(vectordb._collection.count())    # same as number of splits

### **Similarity Search in Vector store**

In vector databases, algorithms for retrieving relevant chunks to a query are often based on **similarity search techniques**, primarily using nearest neighbor search.

Here are some common approaches:

> **Approximate Nearest Neighbor (ANN) Search:** Vector databases frequently use ANN algorithms to improve efficiency when searching for vectors that
are close to the query vector.
>
> Popular **ANN** algorithms include:

> 1. HNSW (Hierarchical Navigable Small World Graph): This is a graph-based approach that finds approximate nearest neighbors using a multi-
layered graph structure.

> 2. Faiss: An open-source library developed by Facebook, which uses various algorithms for fast similarity search, such as Product Quantization and
Inverted File System (IVF).

> 3. Annoy (Approximate Nearest Neighbors Oh Yeah): Developed by Spotify, it uses a forest of random projection trees for approximate nearest
neighbor search.


In [ ]:
question = "What colors are available?"

In [ ]:
# Perform Similarity Search and fetch top-k chunks

docs = vectordb.similarity_search(question, k=6)     # k --> No. of Document object to return

print(f"Chunks retrieved: {len(docs)}")

In [ ]:
for i in range(len(docs)):
    print(f"CHUNK {i+1}")
    print(docs[i].page_content)
    print('='*140)

### **Edge cases where failure may happen**

1. Lack of Diversity : Semantic search fetches all similar documents, but does not enforce diversity.

    - Notice that we're getting duplicate chunks (because of the duplicate `Product XSound Pro Headphones.pdf` in the index). `docs[0]` and `docs[1]` are indentical.

  **Addressing Diversity - MMR (Maximum Marginal Relevance)**

Maximum Marginal Relevance (MMR) is a method used to retrieve relevant items to a query while avoiding redundancy. It does this by ensuring a balance between relevancy and diversity in the items retrieved.

<img src='https://miro.medium.com/v2/resize:fit:828/format:webp/1*U-9mPt5tBfPBPrwC4_oD1w.png'>

**Without MMR**

The retriever selects the most similar results, which can often be redundant or highly similar to each other.

In [ ]:
# Retrieve chunks without MMR

question = "What colors are available?"
docs = vectordb.similarity_search(question, k=3)     # Without MMR

print(f"Chunks retrieved: {len(docs)}\n")

for i in range(len(docs)):
    print(f"CHUNK {i+1}")
    print(docs[i].page_content)
    print('='*140)

**Example 1. Addressing Diversity - MMR-Maximum Marginal Relevance**

**With MMR**

The retriever balances relevance and diversity, selecting results that are both relevant to the query and different from one another.

In [ ]:
# Retrieve chunks with MMR

docs_with_mmr = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=6)   # With MMR

print(f"Chunks retrieved: {len(docs_with_mmr)}\n")

for i in range(len(docs_with_mmr)):
    print(f"CHUNK {i+1}")
    print(docs_with_mmr[i].page_content)
    print('='*140)

2. Lack of specificity:  The question may be from a particular doc but answer may contain information from other doc.

**Example 2. Addressing Specificity: Working with metadata**

In [ ]:
# Without metadata information / filtering
question = "Whom to reach out if any concerns?"

docs = vectordb.similarity_search(question, k=5)

for doc in docs:
    print({'page': doc.metadata['page'], 'source': doc.metadata['source']})    # metadata contains information about from which doc the answer has been fetched

We can filter the results based on metadata.

In [ ]:
# With metadata information / filtering
question = "Whom to reach out if any concerns?"
docs = vectordb.similarity_search(
    question,
    k=5,
    filter={"source":'/content/Product XSound Pro Headphones.pdf'}     # manually passing metadata, using metadata filter.
)

for doc in docs:
    print({'page': doc.metadata['page'], 'source': doc.metadata['source']})

In [ ]:
# With metadata information + MMR

docs_with_mmr = vectordb.max_marginal_relevance_search(question,
                                                       k=2,
                                                       fetch_k=5,
                                                       filter={"source":'/content/Product XSound Pro Headphones.pdf'}     # manually passing metadata, using metadata filter.
                                                       )

In [ ]:
for i in range(len(docs_with_mmr)):
    print(f"CHUNK {i+1}")
    print(docs_with_mmr[i].page_content)
    print('='*140)

## **Retrieval**

The below code demonstrates how a query (“What colors are available?”) is passed to the vector database retriever, which returns the top **k=3 most relevant document chunks** based on similarity search; these retrieved documents can then be used as context for generating a response.


In [ ]:
# Without MMR
question = "What colors are available?"
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(question)
docs

In [ ]:
# With MMR
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k":5})
docs = retriever.invoke(question)
docs

## **Augmentation**

In [ ]:
from langchain_core.prompts import PromptTemplate                               # To format prompts
from langchain_core.output_parsers import StrOutputParser                       # to transform the output of an LLM into a more usable format
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough     # Required by LCEL (LangChain Expression Language)

In [ ]:
# Build prompt
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""

QA_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)

## **Creating final RAG Chain**

> <img src='https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2F63f8a8482c9ec06a8d7d1041514f87c06dd108a9-3442x942.png&w=3840&q=75' width=1200px>

[[Image source](https://www.pinecone.io/learn/series/langchain/langchain-expression-language/)]

Above figure describes the LCEL flow using `RunnableParallel` and `RunnablePassthrough`.

A Runnable is a **unit of execution** in the LangChain framework. It represents a specific task or operation that can be performed.

Examples of Runnables include data transformations, computations, or any other operation that can be **expressed** in the LCEL(LangChain expression language).

[Runnable Lambdas](https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.base.RunnableLambda.html) is a LangChain abstraction that allows us to turn Python functions into **pipe-compatible functions**, similar to the Runnable class.

[RunnablePassthrough](https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.passthrough.RunnablePassthrough.html) on its own allows you to pass inputs unchanged. This typically is **used in conjuction with [RunnableParallel](https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.base.RunnableParallel.html)** to pass data through to a new key in the map.

The **RunnableParallel** object allows us to define multiple values and operations, and run them all in parallel.

The **RunnablePassthrough** object is used as a “passthrough” that takes any input to the current component ('retrieval' in above figure) and allows us to provide it in the component output via the “question” key or any other custom key.

In [ ]:
# Function to get the context info

def get_context_info(question):
    retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k":5})
    docs = retriever.invoke(question)
    return docs


In [ ]:
# Convert the Python function into pipe-compatible functions

from langchain_core.runnables import RunnableLambda

retrieval = RunnableParallel(
    {
        "context": RunnableLambda(lambda x: get_context_info(x["question"])),
        "question": RunnableLambda(lambda x: x["question"])
        }
    )

In [ ]:
from pprint import pprint

# Test the Retrieval
pprint(retrieval.invoke({"question": "What colors are available?"}))

In [ ]:
# Test the Retrieval
pprint(retrieval.invoke({"question": "When will the salary be credited"}))

**RAG Chain**

In [ ]:
# RAG Chain

rag_chain = (retrieval                     # Retrieval
             | QA_PROMPT                   # Augmentation
             | llm                         # Generation
             | StrOutputParser()
             )

In [ ]:
# Test the RAG Chain
response = rag_chain.invoke({"question": "What colors are available?"})

print(response)

In [ ]:
# Test the RAG Chain
response = rag_chain.invoke({"question": "When will the salary be credited"})

print(response)

In [ ]:
# Test the RAG Chain
response = rag_chain.invoke({"question": "What is the leave policy?"})

print(response)

In [ ]:
# For queries that is not in documents

response = rag_chain.invoke({"question": "Who is the CEO of OpenAI "})

print(response)

## Reusing the Vector DB

The below code demonstrates how to **reuse an existing vector database** in Colab by

* first compressing and downloading the vector store directory, and
* later uploading and extracting it to avoid rebuilding embeddings;
* the Chroma database is then reloaded using the same embedding model, enabling efficient retrieval without recomputing the entire pipeline.


### **Download the Vector DB**

In [ ]:
# Zip the entire folder
!zip -r /content/docs.zip /content/docs

In [ ]:
# Download the Zip file

from google.colab import files
files.download("/content/docs.zip")

### **Upload the vector db from previous step and unzip**

In [ ]:
# Remove the /docs directory if it exists
!rm -r docs

# Unzip the docs.zip file
!unzip /content/docs.zip  -d /

In [ ]:
# Initialize the Vector DB

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding =  HuggingFaceEmbeddings(
    model_name="mixedbread-ai/mxbai-embed-large-v1",                             # Provide the pre-trained model's path
    model_kwargs={'device': "cuda" if torch.cuda.is_available() else "cpu"},     # Pass the model configuration options
    encode_kwargs={'normalize_embeddings': False}                                # Pass the encoding options
)

vectordb = Chroma(persist_directory = 'docs/chroma/',
                  embedding_function = embedding
                  )

In [ ]:
# Retriever

question = "What colors are available?"
new_retriever = vectordb.as_retriever(search_kwargs={"k": 3})
docs = new_retriever.invoke(question)

pprint(docs)

---

$$END$$

---